# create_estdata.ipynb

This notebook creates the input parquet needed for the US logit model. Starting from `pums/pums_{year}.parquet` (built by `pums/unify_pums.ipynb`, which already has `STAY`/`ORIGIN`/`CHOSEN`), it merges in origin-side (MIGPUMA) census (ACS+LODES) and CBSA data, draws `num_alternatives` random destination-PUMA alternatives per person (always including the true `CHOSEN` PUMA for movers), and fills each alternative with census/CBSA/distance/travel-time/own-industry-job data.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from lib import io as lio

In [ ]:
year = 2018
num_alternatives = 100
pums_file = Path("pums/pums_100_2018.parquet")
UNIT_SUFFIXES = ["_REF", "_SEC"]

In [ ]:
pums = pd.read_parquet(pums_file)
pums

,REF_INDEX,SEC_INDEX,NUM_CHILDREN_UNDER_6,NUM_CHILDREN_6_TO_17,SIZE,NUM_IN_IF,NUM_WORKING,PERWT,CHOSEN,ORIGIN,...,AAPI_REF,OTHER_RACE_REF,RACE_ETHNICITY_REF,LATINO_SEC,WHITE_SEC,BLACK_SEC,INDIAN_SEC,AAPI_SEC,OTHER_RACE_SEC,RACE_ETHNICITY_SEC
UNIT,,,,,,,,,,,,,,,,,,,,,
2018000784819_P,2018000784819001,2018000784819001,0,0,1,0,0,150.0,4202703,4202700,...,0,0,1,0,1,0,0,0,0,1
2018000864735_I5,2018000864735005,2018000864735005,0,0,1,0,0,103.0,1800101,1800100,...,0,0,1,0,1,0,0,0,0,1
2018000350562_P,2018000350562001,2018000350562001,0,0,1,1,1,98.0,0400126,0400100,...,0,0,1,0,1,0,0,0,0,1
2018000734419_I3,2018000734419003,2018000734419003,0,0,1,0,0,76.0,4203701,4203700,...,0,0,1,0,1,0,0,0,0,1
2018001070125_P,2018001070125001,2018001070125002,0,3,5,1,1,29.0,2500703,2500390,...,0,0,1,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018000832514_P,2018000832514001,2018000832514002,0,2,4,3,3,264.0,5500103,5500104,...,0,0,99,1,0,0,0,0,0,99
2018010137222_I1,2018010137222001,2018010137222001,0,0,1,1,0,122.0,1702601,1702601,...,0,0,2,0,0,1,0,0,0,2
2018000140762_P,2018000140762001,2018000140762001,0,0,1,1,1,49.0,3601204,3601200,...,0,0,1,0,1,0,0,0,0,1


In [4]:
acs_puma = (
    pd.read_csv(f"acs/acs_puma_{year}.csv", dtype={"PUMA": str})
    .assign(PUMA=lambda d: d["PUMA"].str.zfill(7))
    .set_index("PUMA")
)
acs_migpuma = (
    pd.read_csv(f"acs/acs_migpuma_{year}.csv", dtype={"MIGPUMA": str})
    .assign(MIGPUMA=lambda d: d["MIGPUMA"].str.zfill(7))
    .set_index("MIGPUMA")
)
lodes_puma = (
    pd.read_csv(f"lodes/wac_puma_{year}.csv", dtype={"PUMA": str})
    .assign(PUMA=lambda d: d["PUMA"].str.zfill(7))
    .set_index("PUMA")
)
lodes_migpuma = (
    pd.read_csv(f"lodes/wac_migpuma_{year}.csv", dtype={"MIGPUMA": str})
    .assign(MIGPUMA=lambda d: d["MIGPUMA"].str.zfill(7))
    .set_index("MIGPUMA")
)

In [5]:
puma_cbsa = (
    pd.read_csv(f"geometry/cbsa/puma_density_{year}.csv", dtype={"GEOID": str})
    .assign(GEOID=lambda d: d["GEOID"].str.zfill(7))
    .set_index("GEOID")
)
migpuma_cbsa = (
    pd.read_csv(f"geometry/cbsa/migpuma_density_{year}.csv", dtype={"GISMATCH": str})
    .assign(GISMATCH=lambda d: d["GISMATCH"].str.zfill(7))
    .set_index("GISMATCH")
)

WEATHER_COLS = ["JAN_AVG_TEMP_C", "JULY_AVG_TEMP_C", "AVG_TOT_PPT_M"]
weather_puma = (
    pd.read_csv("weather/puma_weather.csv", dtype={"PUMA": str})
    .assign(PUMA=lambda d: d["PUMA"].str.zfill(7))
    .set_index("PUMA")[WEATHER_COLS]
)
weather_migpuma = (
    pd.read_csv("weather/migpuma_weather.csv", dtype={"MIGPUMA": str})
    .assign(MIGPUMA=lambda d: d["MIGPUMA"].str.zfill(7))
    .set_index("MIGPUMA")[WEATHER_COLS]
)

# PUMA<->MIGPUMA equivalency table (already indexed by PUMA, State already zero-padded
# as it's read as dtype=str) -- gives us each alternative PUMA's state, for ALTi_STATE
puma_migpuma = lio.load_puma_migpuma("geometry/equivalencies/puma_migpuma_2010.csv")
migpuma_to_num_pumas = puma_migpuma.groupby("MIGPUMA").size()
puma_migpuma["MIGPUMA_NUM_PUMAS"] = puma_migpuma["MIGPUMA"].map(migpuma_to_num_pumas)

In [6]:
puma_est = (
    pd.read_csv(f"cbp/puma_est_{year}.csv", dtype={"PUMA": str})
    .assign(PUMA=lambda d: d["PUMA"].str.zfill(7))
    .set_index("PUMA")
    .add_suffix("_NUM_EST")
)
migpuma_est = (
    pd.read_csv(f"cbp/migpuma_est_{year}.csv", dtype={"MIGPUMA": str})
    .assign(MIGPUMA=lambda d: d["MIGPUMA"].str.zfill(7))
    .set_index("MIGPUMA")
    .add_suffix("_NUM_EST")
)

In [7]:
# encode CBSA type (T34/Metro/Micro/other) numerically, and build a single CBSA-name
# codebook shared between the PUMA and MIGPUMA extracts (factorized on the PUMA side,
# since alternatives are PUMA-keyed) so ALTi_CBSA and CBSA_ORIG are directly comparable.
def encode_cbsa_type(df):
    out = np.full(len(df), 3)
    out = np.where(df["TYPE"] == "Nonmetro", 2, out)
    out = np.where(df["TYPE"] == "Metro", 1, out)
    out = np.where(df["TYPE"] == "T34", 0, out)
    df["TYPE_NUM"] = out
    return df


puma_cbsa = encode_cbsa_type(puma_cbsa)
migpuma_cbsa = encode_cbsa_type(migpuma_cbsa)

puma_cbsa["NAME_NUM"], cbsa_name_categories = pd.factorize(puma_cbsa["CBSA_NAME"])
cbsa_name_to_num = dict(zip(cbsa_name_categories, range(len(cbsa_name_categories))))
migpuma_cbsa["NAME_NUM"] = (
    migpuma_cbsa["CBSA_NAME"].map(cbsa_name_to_num).fillna(-2).astype(int)
)

In [8]:
# merge ACS + LODES + CBSA (density/type) + weather (climate normals) into a single
# per-geography table, replacing the old pre-combined census/puma_{year}.csv.
CBSA_COLS = ["CBSA_NAME", "NAME_NUM", "TYPE", "TYPE_NUM"]
puma_data = (
    acs_puma.join(lodes_puma, how="left")
    .join(puma_cbsa[CBSA_COLS].rename_axis("PUMA"), how="left")
    .join(weather_puma, how="left")
    .join(puma_est, how="left")
)
migpuma_data = (
    acs_migpuma.join(lodes_migpuma, how="left")
    .join(migpuma_cbsa[CBSA_COLS].rename_axis("MIGPUMA"), how="left")
    .join(weather_migpuma, how="left")
    .join(migpuma_est, how="left")
)

In [9]:
for data in [puma_data, migpuma_data]:
    data["ENT_EST_PER_CAPITA"] = data["ENT_NUM_EST"] / data["TOT_POP"]
    data["FOD_EST_PER_CAPITA"] = data["FOD_NUM_EST"] / data["TOT_POP"]
    data["AMENITIES_EST_PER_1K_PEOPLE"] = (
        data["ENT_EST_PER_CAPITA"] + data["FOD_EST_PER_CAPITA"]
    ) * 1_000
    data["JOBS_PER_CAPITA"] = data["Total number of jobs"] / data["TOT_POP"]

In [ ]:
migpuma_data_int = migpuma_data.copy()
migpuma_data_int.index = migpuma_data_int.index.astype(int)

pums["ORIGIN_INT"] = pums["ORIGIN"].astype(int)

df = pums.set_index("ORIGIN_INT").join(
    # full ACS+LODES+CBSA+weather columns, now merged into a single table
    migpuma_data_int.add_suffix(".ORIG"),
    how="left",
)
df

In [ ]:
# map each person's raw NAICSP code to a NAICS sector abbreviation, needed for OWN_JOB
# below. ACS's NAICSP industry codes are structured so their first two characters
# already identify the standard 2-digit NAICS sector (e.g. "5411" -> sector 54,
# Professional Services), including two merged "not specified" codes ("3M"/"4M" for
# unspecified Manufacturing/Retail). This replaces the old per-year
# naics_to_lodes_{year}.txt crosswalk, which hand-maintained a label per detailed code
# and had at least one real bug (the 2017 file mislabeled all of NAICS 53 Real Estate as
# FIN instead of REL).
#
# NOTE: NAICS sector 92 (Public Administration) includes military
NAICS_SECTOR_PREFIXES = {
    "11": "AGR",
    "21": "EXT",
    "22": "UTL",
    "23": "CON",
    "31": "MFG",
    "32": "MFG",
    "33": "MFG",
    "3M": "MFG",
    "42": "WHL",
    "44": "RET",
    "45": "RET",
    "4M": "RET",
    "48": "TRN",
    "49": "TRN",
    "51": "INF",
    "52": "FIN",
    "53": "REL",
    "54": "PRF",
    "55": "MNG",
    "56": "ADM",
    "61": "EDU",
    "62": "MED",
    "71": "ENT",
    "72": "FOD",
    "81": "SRV",
    "92": "PUB",
    # "99" corresponds to other/unemployed
}

for suffix in UNIT_SUFFIXES:
    df[f"NAICS{suffix}"] = df[f"INDNAICS{suffix}"].str[0:2].map(NAICS_SECTOR_PREFIXES)
    # people in active service will be treated specially, exclude from PUB
    df[f"NAICS{suffix}"] = np.where(
        df[f"IN_MILITARY{suffix}"], np.nan, df[f"NAICS{suffix}"]
    )
print(df["NAICS_REF"].value_counts(dropna=False))
print(df["NAICS_SEC"].value_counts(dropna=False))

In [ ]:
for suffix in UNIT_SUFFIXES:
    df[f"NAICS_MED{suffix}"] = np.where(df[f"NAICS{suffix}"] == "MED", 1, 0)
    df[f"NAICS_MFG{suffix}"] = np.where(df[f"NAICS{suffix}"] == "MFG", 1, 0)
    df[f"NAICS_RET{suffix}"] = np.where(df[f"NAICS{suffix}"] == "RET", 1, 0)
    df[f"NAICS_EDU{suffix}"] = np.where(df[f"NAICS{suffix}"] == "EDU", 1, 0)
    df[f"NAICS_ADM{suffix}"] = np.where(df[f"NAICS{suffix}"] == "ADM", 1, 0)
    df[f"NAICS_FOD{suffix}"] = np.where(df[f"NAICS{suffix}"] == "FOD", 1, 0)
    df[f"NAICS_PRF{suffix}"] = np.where(df[f"NAICS{suffix}"] == "PRF", 1, 0)
    df[f"NAICS_TRN{suffix}"] = np.where(df[f"NAICS{suffix}"] == "TRN", 1, 0)
    df[f"NAICS_SRV{suffix}"] = np.where(df[f"NAICS{suffix}"] == "SRV", 1, 0)
    df[f"NAICS_FIN{suffix}"] = np.where(df[f"NAICS{suffix}"] == "FIN", 1, 0)
    df[f"NAICS_WHL{suffix}"] = np.where(df[f"NAICS{suffix}"] == "WHL", 1, 0)
    df[f"NAICS_AGR{suffix}"] = np.where(df[f"NAICS{suffix}"] == "AGR", 1, 0)
    df[f"NAICS_PUB{suffix}"] = np.where(df[f"NAICS{suffix}"] == "PUB", 1, 0)
    df[f"NAICS_INF{suffix}"] = np.where(df[f"NAICS{suffix}"] == "INF", 1, 0)
    df[f"NAICS_ENT{suffix}"] = np.where(df[f"NAICS{suffix}"] == "ENT", 1, 0)
    df[f"NAICS_REL{suffix}"] = np.where(df[f"NAICS{suffix}"] == "REL", 1, 0)
    df[f"NAICS_UTL{suffix}"] = np.where(df[f"NAICS{suffix}"] == "UTL", 1, 0)
    df[f"NAICS_EXT{suffix}"] = np.where(df[f"NAICS{suffix}"] == "EXT", 1, 0)
    df[f"NAICS_MNG{suffix}"] = np.where(df[f"NAICS{suffix}"] == "MNG", 1, 0)
    df[f"NAICS_CON{suffix}"] = np.where(df[f"NAICS{suffix}"] == "CON", 1, 0)
    df[f"NAICS_OTHER{suffix}"] = np.where(df[f"NAICS{suffix}"].isna(), 1, 0)

In [ ]:
# combine NAICS job categories into groups, by skill/credential barrier to entry,
# for use in migration model. Shared by NAICS_{group} (this cell, individual-level
# one-hot dummies) and NAICS_GROUP_PROP_{group} (added onto puma_data/migpuma_data
# below, area-level job shares) so the two stay in sync.
NAICS_GROUPS = {
    # primary/extractive: resource-tied to location
    "AGR_EXT": ["AGR", "EXT"],
    # high-education professional: bachelor's+ degree typical, portable credentials
    "HIGH_ED": ["MED", "EDU", "PRF", "FIN", "INF", "MNG"],
    # occupational-license-heavy services: state-specific licenses (real estate,
    # personal care/repair, etc.), a classic migration friction
    "LICENSE": ["SRV", "REL"],
    # goods-producing/trade: apprenticeship or on-the-job skill, not degree/license driven
    "GOODS_TRADE": ["MFG", "CON", "WHL", "TRN", "UTL"],
    # low-skill consumer services: low-license, low-credential
    "LOW_SKILL_SVC": ["RET", "FOD", "ADM", "ENT"],
    # public administration
    "GOVT": ["PUB"],
}

for suffix in UNIT_SUFFIXES:
    for group, sectors in NAICS_GROUPS.items():
        # max on [0, 1] is effectively an or
        df[f"NAICS_{group}{suffix}"] = df[[f"NAICS_{s}{suffix}" for s in sectors]].max(
            axis=1
        )

In [ ]:
EDU_EARNINGS_RAW_COLS = {
    "EDU_NOHIGH": "Median earnings in tens of thousands of dollars for people without high school diplomas",
    "EDU_ONLY_HIGH": "Median earnings in tens of thousands of dollars for high school graduates only",
    "EDU_SOME_COLLEGE": "Median earnings in tens of thousands of dollars for people with some college only",
    "EDU_ONLY_BACHELORS": "Median earnings in tens of thousands of dollars for people with bachelors only",
    "EDU_GRADUATE_DEG": "Median earnings in tens of thousands of dollars for people with graduate degrees",
}
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = 0
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = np.where(
    df["EDU_NOHIGH"],
    df[EDU_EARNINGS_RAW_COLS["EDU_NOHIGH"] + ".ORIG"],
    df["OWN_EARNINGS_10K_BY_EDU.ORIG"],
)
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = np.where(
    df["EDU_ONLY_HIGH"],
    df[EDU_EARNINGS_RAW_COLS["EDU_ONLY_HIGH"] + ".ORIG"],
    df["OWN_EARNINGS_10K_BY_EDU.ORIG"],
)
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = np.where(
    df["EDU_SOME_COLLEGE"],
    df[EDU_EARNINGS_RAW_COLS["EDU_SOME_COLLEGE"] + ".ORIG"],
    df["OWN_EARNINGS_10K_BY_EDU.ORIG"],
)
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = np.where(
    df["EDU_ONLY_BACHELORS"],
    df[EDU_EARNINGS_RAW_COLS["EDU_ONLY_BACHELORS"] + ".ORIG"],
    df["OWN_EARNINGS_10K_BY_EDU.ORIG"],
)
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = np.where(
    df["EDU_GRADUATE_DEG"],
    df[EDU_EARNINGS_RAW_COLS["EDU_GRADUATE_DEG"] + ".ORIG"],
    df["OWN_EARNINGS_10K_BY_EDU.ORIG"],
)
assert df["OWN_EARNINGS_10K_BY_EDU.ORIG"].min() > 0

In [ ]:
df["ORIGIN_NUM_PUMAS"] = df["ORIGIN"].map(puma_migpuma["MIGPUMA_NUM_PUMAS"])

In [15]:
# current origin(MIGPUMA)->destination(PUMA) distance/time matrices
distance_matrix = lio.load_distance_matrix(
    "distances/puma_migpuma_distance_matrix.csv", zfill=7
)
time_matrix = lio.load_distance_matrix(
    "distances/puma_migpuma_time_matrix.csv", zfill=7
)
alt_pool = np.array(distance_matrix.columns, dtype="<U7")
len(alt_pool)

2336

In [ ]:
# draw num_alternatives random destination PUMAs per person (vectorized, in row-batches to
# bound peak memory).
# For movers (STAY==0), slot 0 is forced to be their actual CHOSEN PUMA so the true choice
# is always present in the sampled choice set exactly once.
rng = np.random.default_rng(5063)  # fixed seed for reproducibility
n = len(df)  # number of PUMS records (rows) to sample alternatives for
pool_size = len(alt_pool)  # total number of candidate PUMAs to sample from

# output array: n rows x num_alternatives columns, each cell a PUMA code string (max 7 chars)
# store POOL POSITIONS (int16), not PUMA code strings. A "<U7" array of this shape is
# 28 bytes/element -- ~4.9 GB at 1.74M x 100 -- against 348 MB for int16. pool_size is
# 2,336, comfortably inside int16. Positions are mapped back to codes one alternative
# at a time in the fill loop below.
assert pool_size < np.iinfo(np.int16).max, "pool too large for int16 positions"
random_pumas = np.empty((n, num_alternatives), dtype=np.int16)

chosen = df["CHOSEN"].to_numpy()  # each person's actual chosen destination PUMA
stay = df["STAY"].to_numpy()  # flag: 1 = stayed in place, 0 = moved to CHOSEN
origin = df["ORIGIN"].to_numpy()  # each person's origin MIGPUMA

# build a lookup from PUMA code -> its integer position within alt_pool, so we can
# work with the sampling in terms of array indices rather than string comparisons
pool_index = {puma: i for i, puma in enumerate(alt_pool)}

# for every record, find where its CHOSEN puma sits in alt_pool (-1 if not present,
# e.g. for stayers where CHOSEN may not be a valid/relevant destination)
chosen_pos = np.array([pool_index.get(p, -1) for p in chosen])

# sanity check: every mover's chosen destination must actually exist in the pool,
# otherwise we'd have no way to force it into slot 0 below
assert (chosen_pos[stay == 0] >= 0).all(), (
    "some movers' CHOSEN puma is missing from the alternative pool"
)

# MIGPUMA that each pool position belongs to, so we can exclude a person's own
# origin region from their sampled alternatives (it's already the stay alternative)
pool_migpuma = pd.Series(alt_pool).map(puma_migpuma["MIGPUMA"]).to_numpy()
assert not pd.isna(pool_migpuma).any(), "some pool PUMAs have no MIGPUMA mapping"
assert pool_migpuma.dtype == origin.dtype, "ORIGIN / MIGPUMA dtype mismatch"

# verify that accounting for migpuma exclusions, there are enough draws for each person
max_banned = pd.Series(pool_migpuma).value_counts().max()
assert num_alternatives <= pool_size - max_banned, "pool too small after exclusions"

# process rows in batches so we never materialize an (n x pool_size) array,
# which could be huge for large PUMS extracts / large pools
batch_size = 25_000
for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    # draw one random uniform "key" per (row, pool-position) pair in this batch;
    # sorting these keys per row gives us a random permutation of pool positions
    # each of these are a value from 0-1
    keys = rng.random((end - start, pool_size))

    # positions inside this person's own origin MIGPUMA get a key above 1.0, so they
    # always sort after every legitimate draw and never reach the first k columns
    banned = pool_migpuma[None, :] == origin[start:end, None]
    keys[banned] = 2.0

    # argsort each row's keys and keep only the first num_alternatives columns:
    # this is equivalent to sampling num_alternatives distinct positions from the
    # pool without replacement, per row, without explicitly shuffling the whole pool
    # the argsort sorts the random values from 0-1, equivalent to a random permutation of all the possible pumas to move to
    # take the first num_alternatives of this permutation to get our desired alternatives
    order = np.argsort(keys, axis=1)[
        :, :num_alternatives
    ]  # random subset of pool positions, per row

    # slice this batch's chosen-position and mover-status arrays to match `order`
    batch_chosen_pos = chosen_pos[start:end]
    movers_mask = stay[start:end] == 0

    # for each row, check whether the CHOSEN position happens to already appear
    # somewhere among the num_alternatives sampled columns
    match = order == batch_chosen_pos[:, None]
    already_present = match.any(axis=1)
    # column index of the first (only, since positions are drawn w/o replacement) match;
    # meaningless for rows where already_present is False, but those get overwritten below anyway
    match_col = np.argmax(match, axis=1)

    # case 1: mover whose CHOSEN puma was NOT among the sampled alternatives ->
    # forcibly overwrite slot 0 with the chosen position, guaranteeing it's included
    need_insert = movers_mask & ~already_present
    order[need_insert, 0] = batch_chosen_pos[need_insert]

    # case 2: mover whose CHOSEN puma WAS sampled, but landed in some column other than 0 ->
    # swap it into slot 0 (rather than overwrite) so it still appears exactly once,
    # just relocated, preserving the rest of the sampled alternatives
    need_swap = movers_mask & already_present & (match_col != 0)
    rows_to_swap = np.nonzero(need_swap)[
        0
    ]  # row indices (within this batch) needing a swap
    cols_to_swap = match_col[need_swap]  # the column each of those rows' match sits in
    tmp = order[rows_to_swap, 0].copy()  # save whatever was in slot 0 before swapping
    order[rows_to_swap, 0] = order[
        rows_to_swap, cols_to_swap
    ]  # move chosen puma into slot 0
    order[rows_to_swap, cols_to_swap] = tmp  # put the displaced value in its place

    # note: stayers (movers_mask == False) are left untouched here — their slot 0
    # is whatever was randomly sampled, since there's no CHOSEN puma to force in

    # translate sampled pool positions back into actual PUMA code strings and store
    random_pumas[start:end] = order.astype(np.int16)

random_pumas.shape

In [ ]:
# sanity checks on the drawn alternatives
movers = stay == 0
# random_pumas holds pool positions, so compare against chosen_pos rather than the
# PUMA code strings
assert (random_pumas[movers, 0] == chosen_pos[movers]).all(), (
    "ALT1 != CHOSEN for some movers"
)
sample_idx = rng.choice(n, size=min(500, n), replace=False)
dup_counts = np.array([len(set(row)) for row in random_pumas[sample_idx]])
assert (dup_counts == num_alternatives).all(), (
    "found a duplicate PUMA within a single person's alternative set"
)

# no drawn alternative may lie inside the person's own ORIGIN MIGPUMA. That region is
# already represented by the stay option, so including one of its PUMAs as a "move"
# alternative would put the same choice in the set twice. The sampler enforces this via
# keys[banned] = 2.0, but slot 0 is overwritten/swapped for movers afterwards, so verify
# the invariant on the finished array rather than trusting the construction.
# Checked column by column so we never materialise an (n x num_alternatives) array.
# pool_migpuma is already indexed by pool position, so this is now a plain gather
# rather than a per-column string map
internal = 0
for _i in range(num_alternatives):
    internal += int((pool_migpuma[random_pumas[:, _i]] == origin).sum())
assert internal == 0, (
    f"{internal} drawn alternatives lie inside their own ORIGIN MIGPUMA"
)


In [ ]:
# curated subset of ACS/LODES fields to bring in (verbose "Category.Subcategory.SE_CODE"
# column names resolved against the current acs_puma_2018.csv/wac_puma_2018.csv headers,
# & their migpuma-side equivalents)
# NOTE: many of these are commented out since they add a significant amount of columns
CENSUS_FIELDS = {
    "TOT_POP": "TOT_POP",
    # "DENS": "Population Density (Per Sq. Mile).Population Density (Per Sq. Mile).SE_A00002_002",
    "FOREIGN_BORN_PROP": "Proportion foreign born",
    "MIL_PROP": "Proportion of people in military",
    # "PUBLIC_ASSISTANCE_PROP": "Proportion of people receiving public assistance income",
    # "MED_EARNINGS_10K": "Median earnings in tens of thousands of dollars",
    "MED_HOUSE_VAL_100k": "Median house cost in hundreds of thousands of dollars",
    "MED_RENT_K": "Median gross rent in thousands of dollars",
    "UNEMP_RATE": "Unemployment rate",
    "COLLEGE_PROP": "Proportion of people in college",
    "HOUSE_VACANCY_PROP": "House vacancy proportion",
    "MED_TRAVEL_TIME": "Median travel time",
    # "MED_RENT_PROP_HH_INC": "Median gross rent as a percentage of household income",
    # "MED_HOUSE_VALUE_OVER_MED_HH_INC": "Median house value over median household income",
    # "POVERTY_PROP": "Proportion of people struggling",
    # "YR_SINCE_MED_STRUCTURE": "Years since median structure built",
    "HH_WITH_CHILD_PROP": "Proportion of households with children",
    # "LF_PARTCP_RATE": "Labor force participation rate",
    # "JOBS_PER_CAPITA": "JOBS_PER_CAPITA",
    # "AMENITIES_EST_PER_1K_PEOPLE": "AMENITIES_EST_PER_1K_PEOPLE",
    "JAN_AVG_TEMP_C": "JAN_AVG_TEMP_C",
    # "JULY_AVG_TEMP_C": "JULY_AVG_TEMP_C",
    "AVG_TOT_PPT_M": "AVG_TOT_PPT_M",
    "ALT_COMMUTE_PROP": "Proportion alternative commute",  # walking or public transit
    # "MED_OWNER_COST_HH_INC_PROP": "Median selected monthly owner costs as percentage of household income",
    "ENT_JOB_PROP": "Proportion of entertainment jobs",
    # "TOT_JOBS": "Total number of jobs",
}
NAICS_SECTOR_COLUMNS = {
    "AGR": "Number of jobs in NAICS sector 11 (Agriculture, Forestry, Fishing and Hunting)",
    "EXT": "Number of jobs in NAICS sector 21 (Mining, Quarrying, and Oil and Gas Extraction)",
    "UTL": "Number of jobs in NAICS sector 22 (Utilities)",
    "CON": "Number of jobs in NAICS sector 23 (Construction)",
    "MFG": "Number of jobs in NAICS sector 31-33 (Manufacturing)",
    "WHL": "Number of jobs in NAICS sector 42 (Wholesale Trade)",
    "RET": "Number of jobs in NAICS sector 44-45 (Retail Trade)",
    "TRN": "Number of jobs in NAICS sector 48-49 (Transportation and Warehousing)",
    "INF": "Number of jobs in NAICS sector 51 (Information)",
    "FIN": "Number of jobs in NAICS sector 52 (Finance and Insurance)",
    "REL": "Number of jobs in NAICS sector 53 (Real Estate and Rental and Leasing)",
    "PRF": "Number of jobs in NAICS sector 54 (Professional, Scientific, and Technical Services)",
    "MNG": "Number of jobs in NAICS sector 55 (Management of Companies and Enterprises)",
    "ADM": "Number of jobs in NAICS sector 56 (Administrative and Support and Waste Management and Remediation Services)",
    "EDU": "Number of jobs in NAICS sector 61 (Educational Services)",
    "MED": "Number of jobs in NAICS sector 62 (Health Care and Social Assistance)",
    "ENT": "Number of jobs in NAICS sector 71 (Arts, Entertainment, and Recreation)",
    "FOD": "Number of jobs in NAICS sector 72 (Accommodation and Food Services)",
    "SRV": "Number of jobs in NAICS sector 81 (Other Services [except Public Administration])",
    "PUB": "Number of jobs in NAICS sector 92 (Public Administration)",
}

AGE_BRACKET_COLS = [
    "Proportion of people under 18",
    "Proportion of people 18-34",
    "Proportion of people 35-64",
    "Proportion of people 65+",
]


"""
RACE_ETHNICITY_{REF,SEC} -- IPUMS RACE, with Hispanic overriding race
1 .White
2 .Black/African American
3 .American Indian or Alaska Native
4 .Chinese
5 .Japanese
6 .Other Asian or Pacific Islander
7 .Other race
8 .Two major races
9 .Three or more major races

99 .Latino/Hispanic (HISPAN != 0, injected upstream)

NOTE: these are IPUMS RACE codes, NOT ACS RAC1P -- the two agree only on 1 and 2.
RAC1P spreads AIAN across 3/4/5 and puts Asian at 6, NHPI at 7, other race at 8,
multiracial at 9. IPUMS puts all AIAN at 3, Chinese/Japanese/other-Asian-or-PI at
4/5/6, other race at 7, and multiracial at 8/9. Keying this dict on RAC1P would
send Chinese and Japanese units to the American Indian population share and
"other race" units to AAPI.
"""

RACE_COLUMNS = {
    1: "Proportion of people White",
    2: "Proportion of people Black",
    3: "Proportion of people Indian",
    4: "Proportion of people AAPI",
    5: "Proportion of people AAPI",
    6: "Proportion of people AAPI",
    7: "Proportion of people other race",
    8: "Proportion of people other race",
    9: "Proportion of people other race",
    99: "Proportion of people Latino",
}

In [19]:
# area-level companion to the NAICS_{group} dummies above: for each PUMA/MIGPUMA, the
# share of all jobs that fall into each of the 6 skill/credential groups (sum of the
# group's raw NAICS_SECTOR_COLUMNS job counts, divided by total jobs). Used below to
# build OWN_JOB/OWN_JOB_ORIG as "the share of jobs at this location in my own group"
# rather than a raw job count in my narrow individual NAICS sector.
NAICS_GROUP_PROP_COLUMNS = [f"NAICS_GROUP_PROP_{group}" for group in NAICS_GROUPS]


def add_naics_group_prop_columns(census_df):
    total_jobs = census_df["Total number of jobs"]
    for group, sectors in NAICS_GROUPS.items():
        group_jobs = census_df[[NAICS_SECTOR_COLUMNS[s] for s in sectors]].sum(axis=1)
        census_df[f"NAICS_GROUP_PROP_{group}"] = np.where(
            total_jobs == 0, 0, group_jobs / total_jobs
        )
    return census_df


puma_data = add_naics_group_prop_columns(puma_data)
migpuma_data = add_naics_group_prop_columns(migpuma_data)

In [20]:
for col in puma_data.columns:
    print(col)

Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).2.00 to 2.99.SE_C13004_003
Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).3.00 to 3.99.SE_C13004_004
Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).4.00 to 4.99.SE_C13004_005
Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).5.00 and Over.SE_C13004_006
Aggregate Family Income (In 2018 Inflation Adjusted Dollars).Aggregate Family Income (In 2018 Inflation Adjusted Dollars).SE_A14020_001
Aggregate Gross Rent.Aggregate Gross Rent for Specified  Renter-Occupied Housing Units.SE_A18004_001
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Two or More Races Householder.SE_A14019_008
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) White Alone Householder.SE_A14019_002
Aggr

In [ ]:
# fill each alternative with census (ACS+LODES+CBSA+weather), distance/time, and
# NAICS-group job-share data. ORIGIN varies per row and the alternative PUMA varies per
# row too, so distance/time lookups are a paired (diagonal) gather -- get positional
# indices and index the underlying numpy arrays directly (DataFrame.lookup() was removed
# in pandas 2.x).
puma_cols = list(CENSUS_FIELDS.values()) + ["TYPE_NUM", "NAME_NUM"]
alt_col_names = list(CENSUS_FIELDS.keys()) + ["TYPE", "CBSA"]

origin_row_pos = distance_matrix.index.get_indexer(df["ORIGIN"])
assert (origin_row_pos >= 0).all(), "some ORIGIN missing from the distance matrix"

# origin (MIGPUMA)-side share of jobs in each NAICS skill/credential group -- a plain
# per-row lookup by ORIGIN, one column per group (rather than gathering just the
# person's own group), so the modeling notebook can weight each group by its own
# coefficient and the person's own NAICS_{group} dummy directly.
origin_group_prop_block = migpuma_data.loc[
    df["ORIGIN"], NAICS_GROUP_PROP_COLUMNS
].reset_index(drop=True)
for col in NAICS_GROUP_PROP_COLUMNS:
    df[col + ".ORIG"] = origin_group_prop_block[col].astype(np.float32).to_numpy()

# person's own age-bracket census column name (constant across alternatives)
assert (
    df[["AGE_UNDER_18", "AGE_18_34", "AGE_35_64", "AGE_OVER_65"]].sum(axis=1) == 1
).all(), "AGE_* dummies on df aren't mutually exclusive/exhaustive"
own_age_col_name = np.select(
    [
        df["AGE_UNDER_18"] == 1,
        df["AGE_18_34"] == 1,
        df["AGE_35_64"] == 1,
        df["AGE_OVER_65"] == 1,
    ],
    AGE_BRACKET_COLS,
    default="",  # unreachable given the exhaustiveness assert above
)

# unit's RACE_ETHNICITY race census column names (constant across alternatives), one
# per member. The gathered shares are averaged across REF/SEC below; for unpaired units
# _SEC duplicates _REF, so the mean is just the reference person's value.
own_race_col_name, own_race_missing = {}, {}
for _sfx in UNIT_SUFFIXES:
    _name = df[f"RACE_ETHNICITY{_sfx}"].map(RACE_COLUMNS).to_numpy()
    own_race_missing[_sfx] = pd.isna(_name)
    own_race_col_name[_sfx] = np.where(
        own_race_missing[_sfx], next(iter(RACE_COLUMNS.values())), _name
    )

# person's own NAICS_{group} column name (constant across alternatives), same idea as
# age/race above: gather just the alternative's job share in *this person's own*
# skill/credential group into a single OWN_GROUP_PROP column, rather than bringing in
# all 6 NAICS_GROUP_PROP_* columns per alternative. People with no group (NAICS_OTHER --
# unemployed/not in labor force/military, see NAICS_GROUPS above) have all 6
# NAICS_{group} dummies at 0, so np.select falls through to the "" default -- those
# members are then skipped by mean_over_present rather than contributing a 0.
own_naics_col_name, own_naics_missing = {}, {}
for _sfx in UNIT_SUFFIXES:
    _name = np.select(
        [df[f"NAICS_{group}{_sfx}"] == 1 for group in NAICS_GROUPS],
        NAICS_GROUP_PROP_COLUMNS,
        default="",
    )
    own_naics_missing[_sfx] = _name == ""
    own_naics_col_name[_sfx] = np.where(
        own_naics_missing[_sfx], NAICS_GROUP_PROP_COLUMNS[0], _name
    )

# Per-member weights for the paired terms downstream. A fixed 0.5 would halve a unit
# whose only categorised member is one of two slots: a couple with a single earner would
# contribute 0.5*p while the SAME earner living alone contributes p, because _SEC
# duplicates _REF for unpaired units. Dividing by the number of members that actually
# have a category restores the nan-mean behaviour --
#   both categorised -> 0.5 each      (their mean)
#   one categorised  -> 1.0 for them  (their full share, undiluted)
#   neither          -> 0             (the term drops out)
for _kind, _missing in (("RACE", own_race_missing), ("NAICS", own_naics_missing)):
    _n_present = sum((~_missing[_s]).astype(np.int8) for _s in UNIT_SUFFIXES)
    for _sfx in UNIT_SUFFIXES:
        df[f"{_kind}_W{_sfx}"] = np.where(
            _missing[_sfx], 0.0, 1.0 / np.maximum(_n_present, 1)
        ).astype(np.float32)


# Origin-side counterparts of the per-alternative OWN_* gathers, kept SEPARATE by member
# rather than averaged into one column. Averaging loses which member sits in which
# category, so the result can no longer be paired against a per-category indicator: a
# Black/White couple would end up scoring half the White share under the Black
# coefficient. Downstream the term for category c is
#     0.5 * IND_c_REF * OWN_*_REF  +  0.5 * IND_c_SEC * OWN_*_SEC
# which equals (unit's fraction in c) x (area's share of c).
# A member with no matching category contributes 0, and their indicator is 0 too, so the
# pair drops out cleanly.
_origin_group_np = origin_group_prop_block.to_numpy()
for _sfx in UNIT_SUFFIXES:
    _pos = origin_group_prop_block.columns.get_indexer(own_naics_col_name[_sfx])
    _v = _origin_group_np[np.arange(n), _pos].astype(np.float32)
    df[f"OWN_NAICS_GROUP_PROP{_sfx}.ORIG"] = np.where(
        own_naics_missing[_sfx], 0, _v
    ).astype(np.float32)

# same for race -- the ORIGIN MIGPUMA's population share in each member's own category.
# Without these the stay alternative cannot be scored on own-race share while every move
# alternative can, so the coefficient would be identified off moves only.
_race_orig_cols = list(dict.fromkeys(RACE_COLUMNS.values()))
_origin_race_block = migpuma_data.loc[df["ORIGIN"], _race_orig_cols].reset_index(
    drop=True
)
_origin_race_np = _origin_race_block.to_numpy()
for _sfx in UNIT_SUFFIXES:
    _pos = _origin_race_block.columns.get_indexer(own_race_col_name[_sfx])
    _v = _origin_race_np[np.arange(n), _pos].astype(np.float32)
    df[f"OWN_RACE_ETH_PROP{_sfx}.ORIG"] = np.where(
        own_race_missing[_sfx], 0, _v
    ).astype(np.float32)

# person's own education-bracket median-earnings census column name (constant across
# alternatives), same idea as age/race/NAICS above: gather just alternative i's median
# earnings for *this person's own* education bracket into a single OWN_EARNINGS_BY_EDU
# column, reusing the same raw ACS columns already brought in per-alt via
# CENSUS_FIELDS' MED_EARNINGS_* entries. EDU_* (SCHL-derived) dummies are mutually
# exclusive/exhaustive like AGE_*, so no missing/masking case is needed.
assert (df[list(EDU_EARNINGS_RAW_COLS)].sum(axis=1) == 1).all(), (
    "EDU_* dummies on df aren't mutually exclusive/exhaustive"
)
own_edu_earnings_col_name = np.select(
    [df[flag] == 1 for flag in EDU_EARNINGS_RAW_COLS],
    list(EDU_EARNINGS_RAW_COLS.values()),
    default="",  # unreachable given the exhaustiveness assert above
)

demo_cols = (
    list(AGE_BRACKET_COLS)
    + list(dict.fromkeys(RACE_COLUMNS.values()))
    + NAICS_GROUP_PROP_COLUMNS
    + list(EDU_EARNINGS_RAW_COLS.values())
)

dm_np = distance_matrix.to_numpy()
# tm_np = time_matrix.to_numpy()

# NOTE: the recurring pattern here is first getting all the relevant columns for all requested
# alt pumas, storing it into *block
# then, an indexer is created that tells for each person, which index is relevant to them (e.g., correct NAICS gruop)
# afterwards, the block is indexed into by pairs of indices saying that for each person, you should use the corresponding relevant index
# then nas are set to 0 for people who don't have a relevant index (e.g., no NAICS group)
alt_blocks = []
for i in range(num_alternatives):
    # map this alternative's pool positions back to PUMA code strings -- one column at
    # a time, so the full (n x num_alternatives) string array is never materialised
    alt_pumas_i = alt_pool[random_pumas[:, i]]
    key = f"ALT{i + 1}_"

    # curated ACS/LODES fields (CENSUS_FIELDS) plus CBSA type/name codes for alternative
    # i's PUMA, one row per person -- same value repeats for people who happened to draw
    # the same alt PUMA. TYPE_NUM/NAME_NUM ride along as ordinary float32 columns here;
    # the purely-integral downcast pass at the end of the notebook restores them to a
    # compact int dtype.
    census_block = (
        puma_data.loc[alt_pumas_i, puma_cols].reset_index(drop=True).astype(np.float32)
    )
    census_block.columns = [key + c for c in alt_col_names]
    alt_blocks.append(census_block)

    # distance/time from each person's own ORIGIN to alternative i's PUMA -- a paired
    # (row, col) gather since both endpoints vary per person, not a simple column select
    col_pos = distance_matrix.columns.get_indexer(alt_pumas_i)
    assert (col_pos >= 0).all(), (
        f"alternative {i} has PUMAs missing from the distance matrix"
    )
    dist = dm_np[origin_row_pos, col_pos]
    # tme = tm_np[origin_row_pos, col_pos]
    alt_blocks.append(
        pd.DataFrame(
            {
                key + "DIST": dist,
                # key + "TIME": tme
            },
            dtype=np.float32,
        )
    )

    # OWN_AGE_PROP / OWN_RACE_PROP / OWN_GROUP_PROP / OWN_EARNINGS_BY_EDU: per-row
    # gather against the age-bracket, race-share, NAICS-group-share, and
    # education-bracket-earnings columns -- i.e. "what share of alternative i's
    # population/jobs is in this person's own age bracket / race category / NAICS
    # skill-credential group" and "what is alternative i's median earnings for this
    # person's own education bracket" (see the pattern blurb above)
    demo_block = puma_data.loc[alt_pumas_i, demo_cols].reset_index(drop=True)
    demo_np = demo_block.to_numpy()

    age_col_pos = demo_block.columns.get_indexer(own_age_col_name)
    own_age_prop = demo_np[np.arange(n), age_col_pos].astype(np.float32)

    # race / NAICS-group shares are gathered once per unit member, then averaged over
    # the members that actually have a matching category (see mean_over_present).
    # Unpaired units have _SEC == _REF by construction, so their mean is the REF value.
    own_race_prop, own_group_prop = {}, {}
    for _sfx in UNIT_SUFFIXES:
        _pos = demo_block.columns.get_indexer(own_race_col_name[_sfx])
        _v = demo_np[np.arange(n), _pos].astype(np.float32)
        own_race_prop[_sfx] = np.where(own_race_missing[_sfx], 0, _v).astype(np.float32)

        _pos = demo_block.columns.get_indexer(own_naics_col_name[_sfx])
        _v = demo_np[np.arange(n), _pos].astype(np.float32)
        own_group_prop[_sfx] = np.where(own_naics_missing[_sfx], 0, _v).astype(
            np.float32
        )

    edu_earnings_col_pos = demo_block.columns.get_indexer(own_edu_earnings_col_name)
    own_earnings_by_edu = demo_np[np.arange(n), edu_earnings_col_pos].astype(np.float32)

    alt_blocks.append(
        pd.DataFrame(
            {
                key + "OWN_AGE_PROP": own_age_prop,
                **{
                    key + f"OWN_RACE_ETH_PROP{_s}": own_race_prop[_s]
                    for _s in UNIT_SUFFIXES
                },
                **{
                    key + f"OWN_NAICS_GROUP_PROP{_s}": own_group_prop[_s]
                    for _s in UNIT_SUFFIXES
                },
                key + "OWN_EARNINGS_10K_BY_EDU": own_earnings_by_edu,
            }
        )
    )

    # the alternative's raw PUMA code itself, so the choice set is recoverable later
    alt_blocks.append(pd.DataFrame({key + "PUMA": alt_pumas_i}))

    # state FIPS code of alternative i's PUMA
    alt_blocks.append(
        pd.DataFrame({key + "STATE": puma_migpuma.loc[alt_pumas_i, "State"].to_numpy()})
    )

len(alt_blocks)

In [22]:
# concat the base dataframe with all alternative blocks once (not per-iteration)
df_final = pd.concat([df.reset_index(drop=True)] + alt_blocks, axis=1)
df_final.shape

(1265450, 4720)

In [23]:
nas = df_final.isna().sum()
for key in dict(nas[nas > 0]):
    print(key)

CITWP
COW
DRAT
DRATX
ENG
FER
GCL
GCM
GCR
JWMNP
JWRIP
JWTR
MARHD
MARHM
MARHT
MARHW
MARHYP
MLPA
MLPB
MLPCD
MLPE
MLPFG
MLPH
MLPI
MLPJ
MLPK
SCHG
WKHP
WKW
WRK
YOEP
DECADE
DRIVESP
ESP
FOD1P
FOD2P
INDP
JWAP
JWDP
LANP
NAICSP
NOP
OC
OCCP
PAOC
POVPIP
POWPUMA
POWSP
RC
SCIENGP
SCIENGRLP
SFN
SFR
VPS
ACR
MRGP
MRGT
TEN
VALP
VEH
FES
FINCP
FPARC
GRNTP
GRPIP
HHT
HINCP
HUPAOC
HUPARC
MULTG
MV
NOC
OCPIP
PARTNER
R18
R65
SMOCP
TAXAMT
WIF
WKEXREL
WORKSTAT
CBSA_NAME.ORIG
NAICS


In [24]:
df_final.dropna(axis=1, inplace=True)

In [25]:
for col in df_final.columns:
    print(col)

RT
SERIALNO
DIVISION
SPORDER
PUMA
REGION
ST
ADJINC
PWGTP
AGEP
CIT
DDRS
DEAR
DEYE
DOUT
DPHY
DREM
HINS1
HINS2
HINS3
HINS4
HINS5
HINS6
HINS7
INTP
LANX
MAR
MIG
MIL
NWAB
NWAV
NWLA
NWLK
NWRE
OIP
PAP
RELP
RETP
SCH
SCHL
SEMP
SEX
SSIP
SSP
WAGP
WKL
ANC
ANC1P
ANC2P
DIS
ESR
HICOV
HISP
MIGPUMA
MIGSP
MSP
NATIVITY
PERNP
PINCP
POBP
PRIVCOV
PUBCOV
QTRBIR
RAC1P
RAC2P
RAC3P
RACAIAN
RACASN
RACBLK
RACNH
RACNUM
RACPI
RACSOR
RACWHT
WAOB
FAGEP
FCITWP
FFODP
FHISP
FINDP
FINTP
FJWDP
FJWMNP
FJWRIP
FLANP
FMARHYP
FMIGSP
FMILPP
FMILSP
FOCCP
FOIP
FPAP
FPERNP
FPINCP
FPOBP
FPOWSP
FRACP
FRELP
FRETP
FSEMP
FSSIP
FSSP
FWAGP
FWKHP
FYOEP
STAY
ORIGIN
CHOSEN
ORIGIN_STATE
NP
TYPE
CHILD_UNDER_6
CHILD_6_TO_17
CHILD
WORK2_MAR
WORK1_MAR
SINGLE_PARENT
EDU_NOHIGH
EDU_HIGH_BUT_NOT_BACHELORS
EDU_BACHELORS_OR_HIGHER
EDU_ONLY_HIGH
EDU_SOME_COLLEGE
EDU_ONLY_BACHELORS
EDU_GRADUATE_DEG
EDU_HAS_DEGREE
EDU_NO_DEGREE
AGE_UNDER_18
AGE_18_34
AGE_35_64
AGE_18_22
AGE_23_29
AGE_30_39
AGE_40_49
AGE_50_64
AGE_OVER_65
FOREIGN
IN_COLLEGE
WOMAN_WITH_CHI

In [ ]:
df_final.to_parquet(f"estdata_{pums_file.stem}_{num_alternatives}.parquet")